In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('/kaggle/input/datasets/phangud/spamcsv/spam.csv')
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [3]:
df.duplicated().sum()

np.int64(415)

In [4]:
df.drop_duplicates(inplace=True)

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df['Category'].value_counts()

Category
ham     4516
spam     641
Name: count, dtype: int64

In [7]:
df_copy = df.copy()

In [8]:
df_copy.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [9]:
df_copy['Category'] = df_copy['Category'].map({'ham':0,'spam':1})

In [10]:
df_copy['has_url'] = df_copy['Message'].str.contains(r'https?://\S+|www\.\S+',case=False,regex=True).astype(int)

In [11]:
df_copy.head()

,Category,Message,has_url
0,0,"Go until jurong point, crazy.. Available only ...",0
1,0,Ok lar... Joking wif u oni...,0
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,0
3,0,U dun say so early hor... U c already then say...,0
4,0,"Nah I don't think he goes to usf, he lives aro...",0


In [12]:
X = df_copy.drop('Category', axis=1)
y = df_copy['Category']

In [13]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [14]:
X_train

,Message,has_url
2598,"Got fujitsu, ibm, hp, toshiba... Got a lot of ...",0
5418,So how are you really. What are you up to. How...,0
99,I see a cup of coffee animation,0
2321,This pain couldn't have come at a worse time.,0
2388,Also where's the piece,0
...,...,...
4750,Thanx u darlin!im cool thanx. A few bday drink...,0
474,Want 2 get laid tonight? Want real Dogging loc...,0
3273,"MOON has come to color your dreams, STARS to m...",0
4022,We have to pick rayan macleran there.,0


In [15]:
X_test

,Message,has_url
3031,"Also sir, i sent you an email about how to log...",0
495,Are you free now?can i call now?,0
2942,My supervisor find 4 me one lor i thk his stud...,0
3911,She.s good. She was wondering if you wont say ...,0
3360,Sorry I missed your call let's talk when you h...,0
...,...,...
2513,"Hiya , have u been paying money into my accoun...",0
1943,K...k:)why cant you come here and search job:),0
3038,Cos darren say ü considering mah so i ask ü...,0
3340,Babe !!!! I LOVE YOU !!!! *covers your face in...,0


In [16]:
import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

nltk.download('stopwords')
nltk.download('wordnet')

stop_words = stopwords.words('english')
lemmatizer = WordNetLemmatizer()

def preprocessing(text):
    text = text.lower()
    text = re.sub('https?://\S+|www.\S+','',text)

    tokens = text.split(' ')

    clean_text = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]

    return " ".join(clean_text)



<>:14: SyntaxWarning: invalid escape sequence '\S'
<>:14: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_1237/3714016230.py:14: SyntaxWarning: invalid escape sequence '\S'
  text = re.sub('https?://\S+|www.\S+','',text)
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [17]:
X_train['cleaned_text'] = X_train['Message'].apply(preprocessing)
X_test['cleaned_text'] = X_test['Message'].apply(preprocessing)

In [18]:
X_train.head()

,Message,has_url,cleaned_text
2598,"Got fujitsu, ibm, hp, toshiba... Got a lot of ...",0,"got fujitsu, ibm, hp, toshiba... got lot model..."
5418,So how are you really. What are you up to. How...,0,really. to. how's masters. on.
99,I see a cup of coffee animation,0,see cup coffee animation
2321,This pain couldn't have come at a worse time.,0,pain come worse time.
2388,Also where's the piece,0,also where's piece


In [19]:
X_train.drop('Message',axis=1)
X_test.drop('Message',axis=1)

,has_url,cleaned_text
3031,0,"also sir, sent email log usc payment portal. i..."
495,0,free now?can call now?
2942,0,supervisor find 4 one lor thk students. havent...
3911,0,she.s good. wondering wont say hi she.s smilin...
3360,0,sorry missed call let's talk time. 07090201529
...,...,...
2513,0,"hiya , u paying money account? so, thanks. got..."
1943,0,k...k:)why cant come search job:)
3038,0,co darren say ü considering mah ask ü...
3340,0,babe !!!! love !!!! *covers face kisses*


In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer(max_features=3000)

In [21]:
X_train_tfidf = tfidf.fit_transform(X_train['cleaned_text'])
X_test_tfidf = tfidf.transform(X_test['cleaned_text'])

In [22]:
from scipy.sparse import hstack


X_train_final = hstack([X_train_tfidf, X_train[['has_url']].values])
X_test_final = hstack([X_test_tfidf, X_test[['has_url']].values])

In [23]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(class_weight='balanced')


In [27]:
model.fit(X_train_final,y_train)

LogisticRegression(class_weight='balanced')

In [30]:
predictions = model.predict(X_test_final)

In [33]:
from sklearn.metrics import classification_report

In [35]:
print(classification_report(predictions,y_test))

              precision    recall  f1-score   support

           0       0.98      0.99      0.98       883
           1       0.95      0.87      0.91       149

    accuracy                           0.97      1032
   macro avg       0.96      0.93      0.95      1032
weighted avg       0.97      0.97      0.97      1032



In [37]:
# Check for duplicate messages between train and test using the cleaned text
train_texts = set(X_train['cleaned_text'])
test_texts = set(X_test['cleaned_text'])

overlap = train_texts.intersection(test_texts)
print(f"Number of duplicate messages shared between train and test: {len(overlap)}")

Number of duplicate messages shared between train and test: 1


In [38]:
from sklearn.naive_bayes import MultinomialNB
nb_model = MultinomialNB()
nb_model.fit(X_train_final,y_train)

MultinomialNB()

In [41]:
predictions = nb_model.predict(X_test_final)

In [42]:
print(classification_report(predictions,y_test))

              precision    recall  f1-score   support

           0       1.00      0.98      0.99       915
           1       0.86      1.00      0.92       117

    accuracy                           0.98      1032
   macro avg       0.93      0.99      0.96      1032
weighted avg       0.98      0.98      0.98      1032



In [45]:
from sklearn.metrics import confusion_matrix, accuracy_score

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

print("Accuracy:", accuracy_score(y_test, predictions))

Confusion Matrix:
[[896   0]
 [ 19 117]]
Accuracy: 0.9815891472868217
